# Create a reproducible two-thirds sample
Randomly select two thirds of the IDs shared by the CSV and GeoJSON, keeping the two output files aligned.

In [1]:
import json
from pathlib import Path
import pandas as pd


In [4]:
folder = Path('.')
csv_path = folder / 'dati_schede.csv'
geojson_path = folder / 'lat_long.json'
output_folder = folder / 'data_reduced'
output_folder.mkdir(parents=True, exist_ok=True)

random_seed = 42

In [5]:
dati = pd.read_csv(csv_path, dtype={'id_scheda_originale': 'string'}, low_memory=False)
with geojson_path.open(encoding='utf-8') as f:
    geojson = json.load(f)

csv_ids = set(dati['id_scheda_originale'].dropna().str.strip())
geojson_ids = {
    str(feature.get('properties', {}).get('ID', '')).strip()
    for feature in geojson['features']
}
shared_ids = sorted((csv_ids & geojson_ids) - {''})

selected_ids = set(
    pd.Series(shared_ids).sample(frac=2/3, random_state=random_seed).tolist()
)

dati_23 = dati[
    dati['id_scheda_originale'].str.strip().isin(selected_ids)
].copy()
dati_23.to_csv(output_folder / 'dati_schede_23.csv', index=False)

features_23 = [
    feature for feature in geojson['features']
    if str(feature.get('properties', {}).get('ID', '')).strip() in selected_ids
]
geojson_23 = {**geojson, 'features': features_23}

with (output_folder / 'lat_long_23.json').open('w', encoding='utf-8') as f:
    json.dump(geojson_23, f, ensure_ascii=False, indent=2)

print(f'Shared IDs: {len(shared_ids)}')
print(f'Selected IDs: {len(selected_ids)}')
print(f'CSV rows kept: {len(dati_23)} / {len(dati)}')
print(f'GeoJSON features kept: {len(features_23)} / {len(geojson["features"])}')
print(f'Output folder: {output_folder.resolve()}')

Shared IDs: 44363
Selected IDs: 29575
CSV rows kept: 29575 / 44363
GeoJSON features kept: 29575 / 44363
Output folder: /mnt/c/Users/gazza/Documents/DOTDOTDOT - summer/FINAL_map/DotDotDot_Rurale_Map/online_map/data/data_reduced
